In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append("./../../")
from src.utils import *
import itertools
import ast


from itertools import chain, combinations

In [62]:
# Load dataset
ds = load_jsonl_file('./../../../data/MATE-dev/mm_0shot_llava_hfllava_1.5_7b_hf.jsonl')
mate_df = pd.read_csv('./../../data/mate_df.csv', index_col=0)


5500it [00:01, 5286.02it/s]


In [61]:
create_mate_df = False
if create_mate_df:
    mate_df = pd.DataFrame(
        {'idx': list(range(0,len(ds))),
        'image': [data['image'] for data in ds],
        'object_count': [data['object_count'] for data in ds],
        'color': [[obj['color'] for obj in data['scene']['objects']] for data in ds],
        'shape': [[obj['shape'] for obj in data['scene']['objects']] for data in ds],
        'material': [[obj['material'] for obj in data['scene']['objects']] for data in ds],
        'color_shape': [[f"{obj['color']}_{obj['shape']}" for obj in data['scene']['objects']] for data in ds],
        'color_material': [[f"{obj['color']}_{obj['material']}" for obj in data['scene']['objects']] for data in ds],
        'shape_material': [[f"{obj['shape']}_{obj['material']}" for obj in data['scene']['objects']] for data in ds],
        'color_shape_material': [[f"{obj['color']}_{obj['shape']}_{obj['material']}" for obj in data['scene']['objects']] for data in ds],
        'size': [[obj['size'] for obj in data['scene']['objects']] for data in ds]}
    )

    mate_df['all'] = mate_df.apply(
        lambda row: row["color"] + row["shape"] + row["material"] +\
                    row["color_shape"] + row["shape_material"] + row["color_material"] + row["color_shape_material"],
        axis=1
    )

    mate_df.to_csv('./../../data/mate_df.csv')

In [69]:
# sizes of object
object_size_ls = set([item for items in mate_df['size'].to_list() for item in ast.literal_eval(items)])
print(f"sizes: {object_size_ls}")

# color of object
color_ls = set([item for items in mate_df['color'].to_list() for item in ast.literal_eval(items)])
print(f"colors: {color_ls}")

# shapes of object
shape_ls = set([item for items in mate_df['shape'].to_list() for item in ast.literal_eval(items)])
print(f"shape: {shape_ls}")

# material of object
material_ls = set([item for items in mate_df['material'].to_list() for item in ast.literal_eval(items)])
print(f"material: {material_ls}")

task_dict = {
    'size': [0.351, 0.701, 0.35, 0.7],
    'color': ['gray', 'yellow', 'red', 'blue', 'brown', 'cyan', 'green', 'purple'],
    'shape': ['sphere', 'cone', 'cube', 'cylinder'],
    'material': ['rubber', 'metal']
}

sizes: {0.351, 0.701, 0.35, 0.7}
colors: {'gray', 'yellow', 'red', 'blue', 'brown', 'cyan', 'green', 'purple'}
shape: {'sphere', 'cone', 'cube', 'cylinder'}
material: {'rubber', 'metal'}


In [10]:
# Number of objects
nobjects = list(range(0,12))

for i in nobjects:
    # Filter dataset for a specific number of objects 
    ds_ = [1 for data in ds if data['object_count'] == i]
    print(f"The number of examples with {i} objects is {len(ds_)}")

The number of examples with 0 objects is 0
The number of examples with 1 objects is 0
The number of examples with 2 objects is 0
The number of examples with 3 objects is 670
The number of examples with 4 objects is 696
The number of examples with 5 objects is 681
The number of examples with 6 objects is 674
The number of examples with 7 objects is 688
The number of examples with 8 objects is 746
The number of examples with 9 objects is 681
The number of examples with 10 objects is 664
The number of examples with 11 objects is 0


In [74]:
# Create list of combinations using color, shape and material with 100+ samples
combinations = []
combinations = combinations + list(itertools.product(task_dict['color']))
combinations = combinations + list(itertools.product(task_dict['shape']))
combinations = combinations + list(itertools.product(task_dict['material']))
combinations = combinations + list(itertools.product(task_dict['color'], task_dict['shape']))
combinations = combinations + list(itertools.product(task_dict['shape'], task_dict['material']))
combinations = combinations + list(itertools.product(task_dict['color'], task_dict['material']))
combinations = combinations + list(itertools.product(task_dict['color'], task_dict['shape'], task_dict['material']))

linear_probe_list_combination = []
linear_probe_list_nobj = []

for nobj in [3,5,7,9]:
    for combination in combinations:
        nsamples = len(mate_df[(mate_df['object_count'] == nobj) & (mate_df['all'].apply(lambda x: '_'.join(combination) in x))])
        if nsamples>99:
            print(f"The number of samples with {combination} objects: {nsamples}, nobjs: {nobj}")
            linear_probe_list_combination = linear_probe_list_combination + ['_'.join(combination)]
            linear_probe_list_nobj = linear_probe_list_nobj + [nobj]


The number of samples with ('gray',) objects: 225, nobjs: 3
The number of samples with ('yellow',) objects: 265, nobjs: 3
The number of samples with ('red',) objects: 257, nobjs: 3
The number of samples with ('blue',) objects: 246, nobjs: 3
The number of samples with ('brown',) objects: 278, nobjs: 3
The number of samples with ('cyan',) objects: 260, nobjs: 3
The number of samples with ('green',) objects: 253, nobjs: 3
The number of samples with ('purple',) objects: 226, nobjs: 3
The number of samples with ('sphere',) objects: 479, nobjs: 3
The number of samples with ('cone',) objects: 276, nobjs: 3
The number of samples with ('cube',) objects: 497, nobjs: 3
The number of samples with ('cylinder',) objects: 505, nobjs: 3
The number of samples with ('rubber',) objects: 585, nobjs: 3
The number of samples with ('metal',) objects: 594, nobjs: 3
The number of samples with ('brown', 'cylinder') objects: 101, nobjs: 3
The number of samples with ('sphere', 'rubber') objects: 262, nobjs: 3
The

In [75]:
linear_probe_df = pd.DataFrame({
    'combinations': linear_probe_list_combination,
    'nobj': linear_probe_list_nobj
})

linear_probe_df

,combinations,nobj
0,gray,3
1,yellow,3
2,red,3
3,blue,3
4,brown,3
...,...,...
260,purple_sphere_rubber,9
261,purple_sphere_metal,9
262,purple_cube_metal,9
263,purple_cylinder_rubber,9


In [18]:
mate_df[(mate_df['object_count']==3) & (mate_df['color_shape'].apply(lambda x: 'gray_cylinder' in x))]

,idx,image,object_count,color,shape,material,color_shape,color_material,shape_material,color_shape_material,size
17,17,3e99992732c50e1307349865b563778d.png,3,"['blue', 'gray', 'yellow']","['cone', 'cylinder', 'cube']","['metal', 'rubber', 'rubber']","['blue_cone', 'gray_cylinder', 'yellow_cube']","['blue_metal', 'gray_rubber', 'yellow_rubber']","['cone_metal', 'cylinder_rubber', 'cube_rubber']","['blue_cone_metal', 'gray_cylinder_rubber', 'y...","[0.351, 0.7, 0.351]"
189,189,bb8dfe0d9814d0e0dcb22fd6c8fc1ad3.png,3,"['gray', 'red', 'yellow']","['cylinder', 'sphere', 'cylinder']","['rubber', 'metal', 'rubber']","['gray_cylinder', 'red_sphere', 'yellow_cylind...","['gray_rubber', 'red_metal', 'yellow_rubber']","['cylinder_rubber', 'sphere_metal', 'cylinder_...","['gray_cylinder_rubber', 'red_sphere_metal', '...","[0.351, 0.351, 0.701]"
211,211,ab8b667a364d57b77cf04656f7e318cc.png,3,"['gray', 'cyan', 'red']","['cylinder', 'cube', 'cone']","['rubber', 'rubber', 'rubber']","['gray_cylinder', 'cyan_cube', 'red_cone']","['gray_rubber', 'cyan_rubber', 'red_rubber']","['cylinder_rubber', 'cube_rubber', 'cone_rubber']","['gray_cylinder_rubber', 'cyan_cube_rubber', '...","[0.351, 0.7, 0.351]"
236,236,867f856462d189cf035207d17ff81d87.png,3,"['blue', 'gray', 'red']","['cylinder', 'cylinder', 'sphere']","['metal', 'metal', 'rubber']","['blue_cylinder', 'gray_cylinder', 'red_sphere']","['blue_metal', 'gray_metal', 'red_rubber']","['cylinder_metal', 'cylinder_metal', 'sphere_r...","['blue_cylinder_metal', 'gray_cylinder_metal',...","[0.701, 0.7, 0.701]"
249,249,abcfb61efc820a8eb42833494a224fde.png,3,"['brown', 'gray', 'purple']","['cube', 'cylinder', 'sphere']","['rubber', 'metal', 'metal']","['brown_cube', 'gray_cylinder', 'purple_sphere']","['brown_rubber', 'gray_metal', 'purple_metal']","['cube_rubber', 'cylinder_metal', 'sphere_metal']","['brown_cube_rubber', 'gray_cylinder_metal', '...","[0.701, 0.701, 0.351]"
...,...,...,...,...,...,...,...,...,...,...,...
5144,5144,21983e4f3b760e07033f13c02724b6d9.png,3,"['blue', 'green', 'gray']","['cube', 'sphere', 'cylinder']","['rubber', 'rubber', 'metal']","['blue_cube', 'green_sphere', 'gray_cylinder']","['blue_rubber', 'green_rubber', 'gray_metal']","['cube_rubber', 'sphere_rubber', 'cylinder_met...","['blue_cube_rubber', 'green_sphere_rubber', 'g...","[0.7, 0.7, 0.351]"
5177,5177,3de1313a059e259ed464c60a5b5b7aa9.png,3,"['red', 'gray', 'green']","['sphere', 'cylinder', 'cube']","['rubber', 'rubber', 'metal']","['red_sphere', 'gray_cylinder', 'green_cube']","['red_rubber', 'gray_rubber', 'green_metal']","['sphere_rubber', 'cylinder_rubber', 'cube_met...","['red_sphere_rubber', 'gray_cylinder_rubber', ...","[0.351, 0.35, 0.351]"
5189,5189,4261cd605b0cf133d040bf438bcb0ed8.png,3,"['yellow', 'gray', 'cyan']","['sphere', 'cylinder', 'cylinder']","['metal', 'rubber', 'metal']","['yellow_sphere', 'gray_cylinder', 'cyan_cylin...","['yellow_metal', 'gray_rubber', 'cyan_metal']","['sphere_metal', 'cylinder_rubber', 'cylinder_...","['yellow_sphere_metal', 'gray_cylinder_rubber'...","[0.7, 0.351, 0.701]"
5295,5295,87bfa5be7db3796562c2f95f762ebd76.png,3,"['gray', 'cyan', 'yellow']","['cylinder', 'cube', 'cone']","['metal', 'metal', 'metal']","['gray_cylinder', 'cyan_cube', 'yellow_cone']","['gray_metal', 'cyan_metal', 'yellow_metal']","['cylinder_metal', 'cube_metal', 'cone_metal']","['gray_cylinder_metal', 'cyan_cube_metal', 'ye...","[0.7, 0.701, 0.7]"
